# Audio Segmentation & Static Masking Pipeline

> **Status (July 2026): not currently in use.** Segmentation moved to a more
> conservative approach that yields shorter clips. This notebook is kept
> working rather than deleted so the lane can be picked back up without
> re-deriving the annotation taxonomy: the category taxonomy, the interval
> arithmetic that keeps masks off speech, and the GCS path convention live in
> `common.audio_masking` and are covered by
> `model/tests/common/tests/test_audio_masking.py`.
>
> **Datasets produced before this fix are wrong and need regenerating.** The
> masking this notebook shipped with had three defects, measured by replaying
> its logic over `broadcastify/feeds/training/manifest.json` (13,515
> annotations, 6,819 exported clips):
>
> | defect | clips affected | total |
> |---|---|---|
> | transcribed audio replaced with pink noise | 280 (4.1%) | 87.2s |
> | unlabeled speech left audible (composite `PII/*` never matched) | 622 (9.1%) | 292.3s |
> | speech attenuated by crossfade ramps | 1,417 (20.8%) | 22.4s |
>
> Between 9% and 13% of clips carried an audio/text misalignment. Both of the
> first two push a model in a specific direction — toward hallucination and
> toward omission respectively — so they are worse than their duration
> suggests. Anything already under `segmented_audio/*_masked/` in GCS should be
> regenerated rather than trusted.
>
> A prototype that extended each clip to the full length of its transmission —
> clustering on every annotated event rather than on speech, and exporting
> speechless transmissions — was explored in
> [PR #733](https://github.com/watch-duty/radio-transcription/pull/733) and set
> aside with the approach. Its analysis of the label taxonomy across 60,533
> manifest entries is what the prefix lists in `common.audio_masking` are built
> from. The clustering changes themselves were never validated against a
> training run and are not carried here.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/common/chirp_and_gemini_segment_masked_audio.ipynb)

This Colab clusters annotated speech into contiguous transmissions, extracts
them, replaces sensitive and untranscribed regions with volume-matched pink
noise, and exports the result to GCS alongside a batch manifest.

### Core Functions

1. **Transmission Building (Clustering)**: Clusters human-voice segments into
   coherent, contiguous "transmissions" based on a **configurable silence gap
   threshold (defaults to 0.5 seconds)**. Only blocks containing a valid
   transcription are retained.
2. **Collision-Aware Padding**: Calculates safe start/end boundaries for each
   extracted transmission. **Dynamic padding is applied (configurable, up to a
   default of 0.5 seconds)** to avoid truncating words, while preventing
   overlap with adjacent transcribed speech.
3. **Category-Driven Masking**: Annotation categories are matched by prefix, so
   composite labels exported by annotators (`PII/ADDRESS`, `PII/ID/PHONE`) are
   classified alongside their base label. `common.audio_masking` sorts every
   label into one of four dispositions:
   - **Speech** (`TRANSCRIPTION*`) — the ground truth; never masked.
   - **Masked** (`PII*`, `UNINTELLIGIBLE`, `FOREIGN_SPEECH`, `UNKNOWN`,
     `LAUGHTER`) — sensitive or untranscribed human voice, replaced with pink
     noise.
   - **Preserved** (`DTMF`, `RINGING`, `STATIC`) — real RF signaling and noise
     floor, left audible so the model learns authentic dispatch audio rather
     than hallucinating speech over it.
   - **Unrecognized** — anything else. The run **stops** rather than exporting
     a label nobody has classified, since it may be speech. Override with
     `ALLOW_UNRECOGNIZED_CATEGORIES` only after inspecting the labels.
4. **Speech Protection**: Masks are expanded by 50ms to swallow click
   transients, then transcribed spans are subtracted from them — so a maskable
   annotation that *spans* an utterance is applied as two sub-masks around it
   rather than over it. The 10ms crossfade ramps are truncated at speech
   boundaries too, so no part of a transcribed span is attenuated.
5. **Automated Manifest & GCS Export**: Writes `.flac` audio under
   `segmented_audio/<dataset>_masked/` and a `batch_manifest.jsonl` mapping
   each segment to its GCS URI, ground truth text, offset, and duration.


In [ ]:
# @title Bootstrap and Install Environment
import os

# Clone repository if not already present
if not os.path.exists("radio-transcription"):
    !git clone -q https://github.com/watch-duty/radio-transcription.git

# Install the model library in editable mode
try:
    import common

    print("✅ Library 'common' already installed.")
except ImportError:
    print("Installing library and dependencies...")
    %pip install -q -e radio-transcription/model

    import site
    import importlib

    importlib.reload(site)

    print("\n✅ Dependencies installed successfully.")

In [ ]:
# @title Install dependencies
%pip install -q --upgrade \
    loguru \
    soundfile \
    colorednoise

In [ ]:
# @title Imports
from collections import defaultdict
import json
from pathlib import Path
import sys
from urllib.parse import urlparse

import colorednoise as cn
from google.cloud import storage
from google.colab import auth, userdata
from IPython.display import display
from loguru import logger
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.notebook import tqdm

from common import audio_masking

In [ ]:
# @title Authentication and client initialization
auth.authenticate_user()


# User Configuration from Colab Secrets
GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GCS_BUCKET = userdata.get("GCS_BUCKET")

!gcloud config set project {GCP_PROJECT_ID} --quiet

# Initialize GCS Client
gcs_client = storage.Client(project=GCP_PROJECT_ID)

In [ ]:
# @title Pipeline Configuration

# fmt: off
# @markdown ### 1. GCS Input & Output Paths
# @markdown Path to the manifest file relative to the 'manifests/' directory (e.g. fire_notifications/eval/manifest.json)
MANIFEST_FILE_PATH = ""  # @param {type:"string"}
# @markdown Path to the audio directory relative to the 'audio/' directory (e.g. fire_notifications/eval)
AUDIO_DIR_PATH = ""  # @param {type:"string"}
# @markdown Dataset path relative to the 'segmented_audio/' directory, WITHOUT a lane suffix (e.g. fire_notifications or echo/eval). The '_masked' suffix is appended for you.
OUTPUT_BASE_PATH = ""  # @param {type:"string"}
# Overwrite existing GCS files?
OVERWRITE_EXISTING = False  # @param {type:"boolean"}

# @markdown ### 2. Segmentation Parameters
# Maximum gap allowed before splitting into a new transmission block (seconds)
TRANSMISSION_GAP_THRESHOLD = 0.5  # @param {type:"slider", min:0.1, max:3.0, step:0.1}
# Collision-Aware Padding (ie, max padding value)
DESIRED_PAD = 0.5  # @param {type:"slider", min:0.0, max:2.0, step:0.1}
# @markdown Export even when the manifest carries labels this pipeline cannot classify. Leave OFF: an unclassified segment is exported unmasked, and it may be speech.
ALLOW_UNRECOGNIZED_CATEGORIES = False  # @param {type:"boolean"}

# @markdown ### 3. System Settings
LOCAL_BASE_PATH = "/content"  # @param {type:"string"}
LOG_LEVEL = "INFO"  # @param ["DEBUG", "INFO", "WARNING", "ERROR"]
# fmt: on

assert MANIFEST_FILE_PATH, (
    "MANIFEST_FILE_PATH must be provided and cannot be empty."
)
assert AUDIO_DIR_PATH, "AUDIO_DIR_PATH must be provided and cannot be empty."
assert OUTPUT_BASE_PATH, (
    "OUTPUT_BASE_PATH must be provided and cannot be empty."
)

# The '_masked' lane suffix is owned by common.audio_masking so that this
# notebook and the inference-manifest notebooks reading its output cannot
# disagree about where a dataset lives. Resolved here so a malformed path
# fails now rather than after the first upload.
GCS_OUTPUT_PREFIX = audio_masking.segmented_audio_prefix(
    OUTPUT_BASE_PATH, masked=True
)

CACHE_DIR = f"{LOCAL_BASE_PATH}/raw_audio"
SEGMENTS_DIR = f"{LOCAL_BASE_PATH}/segmented_audio_masked"
BATCH_MANIFEST_FILENAME = "batch_manifest.jsonl"

# Initialize loguru
logger.remove()
logger.add(
    sys.stderr, level=LOG_LEVEL, format="<level>{level}</level>: {message}"
)

Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
Path(SEGMENTS_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
# @title Transmission Building & Static Masking Logic

# Masks are widened by this much either side to swallow unannotated click
# transients at the edges of an annotation.
MASK_TRANSIENT_PAD_S = 0.05
# Linear crossfade ramp mixed around each mask so the pink noise substitution
# does not click. Truncated at speech boundaries -- see audio_masking.fade_bounds.
CROSSFADE_S = 0.01


def is_speech(segment: dict) -> bool:
    """True when a manifest segment carries ground truth transcription."""
    return (
        audio_masking.classify(segment.get("category"))
        is audio_masking.Disposition.SPEECH
    )


def is_maskable(segment: dict) -> bool:
    """True when a segment's audio should be replaced with pink noise."""
    return (
        audio_masking.classify(segment.get("category"))
        is audio_masking.Disposition.MASK
    )


def is_human_voice(segment: dict) -> bool:
    """True for any human vocalization, transcribed or not.

    These are the segments transmissions are clustered on. Automated
    signaling (DTMF, pager tones) and the noise floor are deliberately
    excluded so they do not bridge two separate transmissions.
    """
    return is_speech(segment) or is_maskable(segment)


def ensure_local_gcs_audio(gcs_uri: str) -> str:
    """Downloads raw audio to local cache and validates file integrity."""
    parsed = urlparse(gcs_uri)
    bucket_name = parsed.netloc
    blob_name = parsed.path.lstrip("/")
    filename = Path(blob_name).name
    local_path = Path(CACHE_DIR) / filename

    # If file exists but is 0 bytes, it's corrupt. Delete it.
    if local_path.exists() and local_path.stat().st_size == 0:
        logger.warning(
            f"Found empty file {filename}, removing for re-download..."
        )
        local_path.unlink()

    if not local_path.exists():
        logger.info(f"Downloading {filename} from GCS...")
        try:
            gcs_client.bucket(bucket_name).blob(blob_name).download_to_filename(
                str(local_path)
            )
        except Exception as e:
            logger.error(f"Failed to download {filename} from GCS: {e}")
            raise e
    return str(local_path)


def cleanup_gcs_output(bucket_name: str, prefix: str) -> None:
    """Deletes the GCS 'directory' (prefix) and all contents recursively."""
    bucket = gcs_client.bucket(bucket_name)
    blobs = list(bucket.list_blobs(prefix=prefix))

    if blobs:
        logger.info(
            f"Deleting prefix and all contents: gs://{bucket_name}/{prefix} ({len(blobs)} files)..."
        )
        bucket.delete_blobs(blobs)
        logger.info("Cleanup complete.")
    else:
        logger.info(
            f"Target prefix gs://{bucket_name}/{prefix} is already empty."
        )


def check_manifest_categories(manifest_data: list[dict]) -> None:
    """Stops the run when the manifest carries unclassified labels.

    An annotation category outside the taxonomy is neither masked nor
    recognized as speech, so its audio would be exported audible and
    untranscribed. That is the failure mode masking exists to prevent, so
    it fails closed by default.
    """
    unrecognized = audio_masking.unrecognized_categories(
        [entry.get("category") for entry in manifest_data]
    )
    if not unrecognized:
        return

    summary = ", ".join(unrecognized)
    if ALLOW_UNRECOGNIZED_CATEGORIES:
        logger.warning(
            f"Exporting with {len(unrecognized)} unclassified annotation "
            f"category/categories: {summary}. Their audio is NOT masked."
        )
        return

    msg = (
        f"Manifest carries annotation categories that common.audio_masking "
        f"does not classify: {summary}. Their audio would be exported "
        f"unmasked. Add them to the taxonomy in "
        f"model/src/common/audio_masking.py, or set "
        f"ALLOW_UNRECOGNIZED_CATEGORIES=True if they are known to be safe."
    )
    raise ValueError(msg)


def build_transmissions(
    anchor_segments: list[dict], threshold: float
) -> list[list[dict]]:
    """Clusters ONLY speech segments into dense transmissions of activity."""
    if not anchor_segments:
        return []

    transmissions = []
    current_transmission = [anchor_segments[0]]
    max_end = anchor_segments[0]["offset"] + anchor_segments[0]["duration"]

    for curr_seg in anchor_segments[1:]:
        gap = curr_seg["offset"] - max_end

        if gap <= threshold:
            current_transmission.append(curr_seg)
            max_end = max(max_end, curr_seg["offset"] + curr_seg["duration"])
        else:
            transmissions.append(current_transmission)
            current_transmission = [curr_seg]
            max_end = curr_seg["offset"] + curr_seg["duration"]

    transmissions.append(current_transmission)
    return transmissions

In [ ]:
# @title Execute Segmentation & Masking Pipeline
def generate_pink_noise(samples: int, volume_scale: float = 0.1) -> np.ndarray:
    """Generates pink noise using the specialized colorednoise library."""
    if samples <= 0:
        return np.array([], dtype=np.float32)
    # beta=1 for pink noise (1/f)
    pink_noise = cn.powerlaw_psd_gaussian(1, samples)

    # 1. Remove DC offset (Center the waveform at zero)
    pink_noise = pink_noise - np.mean(pink_noise)

    # 2. Normalize and scale
    if np.max(np.abs(pink_noise)) > 0:
        pink_noise = (pink_noise / np.max(np.abs(pink_noise))) * volume_scale
    return pink_noise.astype(np.float32)


def apply_pink_noise_mask(
    segment_audio: np.ndarray,
    mask: tuple[int, int],
    protected: list[tuple[int, int]],
    sr: int,
) -> None:
    """Replaces one sample range with volume-matched pink noise, in place.

    Args:
        segment_audio: Extracted segment, modified in place.
        mask: Half-open sample range to replace, with transcribed spans
            already subtracted.
        protected: Transcribed spans, used to keep the crossfade ramps from
            attenuating speech next to the mask.
        sr: Sample rate in Hz.
    """
    m_start, m_end = mask
    if m_end <= m_start:
        return

    # 1. Calculate RMS before we mute anything
    original_clip = segment_audio[m_start:m_end].copy()
    rms_volume = (
        np.sqrt(np.mean(original_clip**2)) if len(original_clip) > 0 else 0.05
    )
    # Clamp the RMS so a loud pop doesn't make the pink noise deafening
    rms_volume = min(0.1, max(rms_volume, 0.04))

    # 2. Crossfade boundaries, truncated where they would touch speech
    f_start, f_end = audio_masking.fade_bounds(
        (m_start, m_end),
        int(sr * CROSSFADE_S),
        protected,
        limit=len(segment_audio),
    )
    fade_in_len = m_start - f_start
    fade_out_len = f_end - m_end

    # 3. HARD MUTE the masked section to kill the transient pop instantly
    segment_audio[m_start:m_end] = 0.0

    # 4. Fade OUT the good audio just BEFORE the mask hits
    if fade_in_len > 0:
        segment_audio[f_start:m_start] *= np.linspace(1, 0, fade_in_len)

    # 5. Fade IN the good audio just AFTER the mask ends
    if fade_out_len > 0:
        segment_audio[m_end:f_end] *= np.linspace(0, 1, fade_out_len)

    # 6. Generate pink noise for the ENTIRE region (fades + mask)
    noise = generate_pink_noise(f_end - f_start, volume_scale=rms_volume * 3.0)

    # 7. Apply crossfades to the noise so it seamlessly blends
    if fade_in_len > 0:
        noise[:fade_in_len] *= np.linspace(0, 1, fade_in_len)
    if fade_out_len > 0:
        noise[-fade_out_len:] *= np.linspace(1, 0, fade_out_len)

    # 8. Mix into the timeline (the masked section is 0.0, so this replaces it)
    segment_audio[f_start:f_end] += noise


def run_segmentation_pipeline() -> None:
    full_output_prefix = GCS_OUTPUT_PREFIX

    # Using existing constants directly
    if OVERWRITE_EXISTING:
        cleanup_gcs_output(GCS_BUCKET, full_output_prefix)

    output_bucket = gcs_client.bucket(GCS_BUCKET)

    full_manifest_path = f"manifests/{MANIFEST_FILE_PATH.lstrip('/')}"
    m_blob = output_bucket.blob(full_manifest_path)
    content = m_blob.download_as_text()

    if not content or len(content.strip()) < 2:
        logger.error(f"Manifest at {full_manifest_path} appears to be empty.")
        return

    try:
        manifest_data = json.loads(content)
    except json.JSONDecodeError as e:
        logger.error(f"Failed to parse manifest: {e}")
        return

    # Fail closed on labels the taxonomy does not cover, before any upload.
    check_manifest_categories(manifest_data)

    files_to_process = defaultdict(list)
    for entry in manifest_data:
        files_to_process[entry["audio_filepath"]].append(entry)

    final_manifest_entries = []

    for json_audio_path, raw_segments in tqdm(
        files_to_process.items(), desc="Processing Audio Files"
    ):
        filename = Path(json_audio_path).name
        example_id = Path(filename).stem
        # Use flexible audio path structure relative to the bucket and audio directory
        true_gcs_uri = (
            f"gs://{GCS_BUCKET}/audio/{AUDIO_DIR_PATH.strip('/')}/{filename}"
        )

        raw_segments.sort(key=lambda x: x["offset"])
        local_src_path = ensure_local_gcs_audio(true_gcs_uri)

        y, sr = sf.read(local_src_path)
        if y.ndim > 1:  # Ensure mono
            y = y.mean(axis=1)

        logger.info(f"Processing {example_id}...")

        # Cluster human voice segments into harvested blocks
        human_segments_only = [
            seg for seg in raw_segments if is_human_voice(seg)
        ]
        clustered_segments = build_transmissions(
            human_segments_only, TRANSMISSION_GAP_THRESHOLD
        )

        # Only keep blocks that contain actual transcription text
        final_segments = [
            block
            for block in clustered_segments
            if any(is_speech(seg) for seg in block)
        ]

        max_audio_time = len(y) / sr

        for i, segment_block in enumerate(final_segments):
            # 1. Extract Ground Truth text
            gt_texts = [
                cleaned
                for seg in segment_block
                if is_speech(seg) and (cleaned := seg.get("text", "").strip())
            ]
            segment_gt_text = " ".join(gt_texts).strip()

            # 2. Identify the core speech boundaries (ignore leading/trailing noise/PII)
            transcription_segs = [s for s in segment_block if is_speech(s)]
            core_start = min(s["offset"] for s in transcription_segs)
            core_end = max(
                s["offset"] + s["duration"] for s in transcription_segs
            )

            # Only check for collisions against other TRANSCRIPTION segments in the full file
            transcription_only_segments = [
                s for s in raw_segments if is_speech(s)
            ]

            # Find closest preceding TRANSCRIPTION label
            preceding_segs = [
                s
                for s in transcription_only_segments
                if s["offset"] + s["duration"] <= core_start
            ]
            available_front_gap = (
                core_start
                - max(s["offset"] + s["duration"] for s in preceding_segs)
                if preceding_segs
                else DESIRED_PAD
            )
            safe_front_pad = max(0.0, min(DESIRED_PAD, available_front_gap))

            # Find closest following TRANSCRIPTION label
            following_segs = [
                s
                for s in transcription_only_segments
                if s["offset"] >= core_end
            ]
            available_back_gap = (
                min(s["offset"] for s in following_segs) - core_end
                if following_segs
                else DESIRED_PAD
            )
            safe_back_pad = max(0.0, min(DESIRED_PAD, available_back_gap))

            # 3. Final Audio Bounds
            segment_start = max(0.0, core_start - safe_front_pad)
            segment_end = min(max_audio_time, core_end + safe_back_pad)

            segment_id = f"{i:03d}"

            out_filename = f"{SEGMENTS_DIR}/{example_id}__seg{segment_id}.flac"
            blob_name = (
                f"{full_output_prefix}/{example_id}/{Path(out_filename).name}"
            )
            output_blob = output_bucket.blob(blob_name)

            if not OVERWRITE_EXISTING and output_blob.exists():
                final_manifest_entries.append(
                    {
                        "audio_filepath": f"gs://{GCS_BUCKET}/{blob_name}",
                        "text": segment_gt_text,
                        "example_id": example_id,
                        "segment_id": segment_id,
                        "offset": segment_start,
                        "duration": segment_end - segment_start,
                    }
                )
                continue

            start_samp = int(segment_start * sr)
            end_samp = int(segment_end * sr)
            segment_audio = y[start_samp:end_samp].copy()

            # Sample ranges of every transcribed span inside this window.
            # Masks and their crossfades are kept off these entirely.
            protected = [
                audio_masking.to_sample_range(
                    t_seg["offset"],
                    t_seg["offset"] + t_seg["duration"],
                    sr,
                    origin_s=segment_start,
                    limit=len(segment_audio),
                )
                for t_seg in transcription_segs
            ]

            for seg in raw_segments:
                if not is_maskable(seg):
                    continue

                raw_mask = audio_masking.to_sample_range(
                    seg["offset"] - MASK_TRANSIENT_PAD_S,
                    seg["offset"] + seg["duration"] + MASK_TRANSIENT_PAD_S,
                    sr,
                    origin_s=segment_start,
                    limit=len(segment_audio),
                )

                # A maskable annotation may span transcribed speech; carve
                # the speech out so the mask lands around it, not over it.
                for sub_mask in audio_masking.subtract_intervals(
                    raw_mask, protected
                ):
                    apply_pink_noise_mask(
                        segment_audio, sub_mask, protected, sr
                    )

            sf.write(
                out_filename, segment_audio, sr, format="FLAC", subtype="PCM_16"
            )
            output_blob.upload_from_filename(out_filename)

            final_manifest_entries.append(
                {
                    "audio_filepath": f"gs://{GCS_BUCKET}/{blob_name}",
                    "text": segment_gt_text,
                    "example_id": example_id,
                    "segment_id": segment_id,
                    "offset": segment_start,
                    "duration": segment_end - segment_start,
                }
            )

    local_manifest = Path(LOCAL_BASE_PATH) / BATCH_MANIFEST_FILENAME
    with open(local_manifest, "w") as f:
        for entry in final_manifest_entries:
            f.write(json.dumps(entry) + "\n")

    output_bucket.blob(
        f"{full_output_prefix}/{BATCH_MANIFEST_FILENAME}"
    ).upload_from_filename(str(local_manifest))
    logger.info(
        f"Pipeline Complete. Exported {len(final_manifest_entries)} segments."
    )

    if final_manifest_entries:
        df = pd.DataFrame(final_manifest_entries)
        display(df[["segment_id", "offset", "duration", "text"]].head(10))

In [ ]:
# @title Create the segments and manifest file
run_segmentation_pipeline()